In [1]:
print("Test")

Test


In [2]:
# Imports
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# Configurations
seed = 42

# Paths
DATA_DIR = pathlib.Path("../data/")
ENC2017_ROOT = DATA_DIR / "enc2017"
UD_ET_EDT_ROOT = DATA_DIR / "ud_et_edt"
HOMONYMS_ROOT = DATA_DIR / "homonymous_word_forms"

ENC2017_DIRS = {
    "processed": ENC2017_ROOT / "processed",
    "raw": ENC2017_ROOT / "raw",
}

UD_ET_EDT_DIRS = {
    "processed": UD_ET_EDT_ROOT / "processed",
    "raw": UD_ET_EDT_ROOT / "raw",
}

HOMONYMS_DIRS = {
    "processed": HOMONYMS_ROOT / "processed",
    "annotations": HOMONYMS_ROOT / "annotations",
}

OUTPUT_DIR = pathlib.Path("../outputs/")

MODEL_DIR = pathlib.Path("../models/")

In [3]:
import json
import estnltk
import tqdm
import pandas as pd
from estnltk.default_resolver import make_resolver

from scripts.model import initialize_model
from scripts.model.bert_morph_tagger import BertMorphTagger

c:\Users\sande\Git_projects\EstNLTK\EstNLTK_model_training\simpletransformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
conflicts_dir = DATA_DIR / "conflicts"
bmt = BertMorphTagger(model_location=str(MODEL_DIR / "NER_mudel_v2"))

In [41]:
# Open JSON file
with open(
    conflicts_dir / "raw" / "nsubj_conflicts_dataset_15022026.json",
    "r",
    encoding="utf-8",
) as f:
    # Load JSON data
    conflicts_data = json.load(f)

In [ ]:
resolver = make_resolver()

In [52]:
for i, row in tqdm.tqdm(enumerate(conflicts_data), total=len(conflicts_data)):
    sentence = row["sentence"]
    word = row["word_form"]
    word_span = row["span"]
    # print(f"Sentence: {sentence}")
    # print(f"Word: {word}, Span: {word_span}")
    text_obj = estnltk.Text(sentence)
    text_obj.tag_layer(["sentences", "words"])
    bmt.tag(text_obj)
    forms = []
    for ma in text_obj.bert_morph_tagging:
        if ma.start == word_span[0] and ma.end == word_span[1]:
            # Extract the form and pos
            for form in ma.form:
                forms.append(form)
    row["bert_morph_v2_form"] = forms

 78%|███████▊  | 7506/9634 [05:47<01:37, 21.84it/s]E:\Git_projects/EstNLTK/EstNLTK_model_training\scripts\model\bert_tokens_to_words_rewriter.py:228: UserWarning: (!) No matching words span for bert token Span('', [{'bert_tokens': '▁»', 'form': '', 'partofspeech': 'Z', 'probability': 0.99999}]).
  warnings.warn(
100%|██████████| 9634/9634 [07:24<00:00, 21.67it/s]


In [ ]:
# Save the updated data back to a new JSON file
with open(
    conflicts_dir
    / "processed"
    / "nsubj_conflicts_dataset_15022026_with_Bert_Morph_v2.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(conflicts_data, f, ensure_ascii=False, indent=2)

In [7]:
# Open the updated JSON file to verify the changes
with open(
    conflicts_dir
    / "processed"
    / "nsubj_conflicts_dataset_15022026_with_Bert_Morph_v2.json",
    "r",
    encoding="utf-8",
) as f:
    conflicts_data = json.load(f)

In [57]:
display(conflicts_data[0])

{'sentence_id': 12410790,
 'sentence': 'Kristina Shmiguni ahistas Holmenkollenis 30 km klassikatehnikasõidu lõpukilomeetritel kurnatus , ent viiendast kohast maailma karikasarja liidrirüü hoidmiseks piisas .',
 'word_form': 'Kristina',
 'span': [0, 8],
 'current_deprel': 'nsubj',
 'current_case': 'gen',
 'current_analysis': 'gen,prop,sg',
 'bert_morph_v2_form': ['sg n']}

In [11]:
# Map the given forms to Vabamorf format
map_forms = {
    "n": "nom",
    "g": "gen",
    "p": "par",
    "adt": "ill",
    "ill": "ill",
    "in": "ine",
    "el": "ela",
    "all": "all",
    "ad": "ade",
    "abl": "abl",
    "tr": "tra",
    "ter": "trm",
    "es": "ess",
    "ab": "abe",
    "kom": "com",
}

In [12]:
def parse_bmt_form(form):
    # If the form is None or empty, return None
    if form is None or form == [""]:
        return None, None
    form_data = form[0].split(" ")
    # If the form is related to a verb
    if len(form_data) == 1:
        word_form = form_data[0]
        return word_form, None
    # Split the form into count and form
    else:
        count = form_data[0]
        word_form = form_data[1]
        # Map the form to Vabamorf format
        mapped_form = map_forms.get(word_form, word_form)
        return word_form, mapped_form

In [80]:
conflicts_data[107]

{'sentence_id': 15887988,
 'sentence': 'Mis ma oskan sul kosta , proovi jah uuemaid draivereid , proovi mälu CAS 2 kui ei aita , siis oled stuck ...',
 'word_form': 'proovi',
 'span': [57, 63],
 'current_deprel': 'nsubj',
 'current_case': 'gen',
 'current_analysis': 'com,gen,sg',
 'bert_morph_v2_form': ['o']}

In [99]:
# Count:
# - the number of matching forms
# - the number of non-matching forms
# - the number of no predicted forms (bert morph v2 form is None)
# - the number of non-matching forms where the Bert Morph v2 form is in ["n", "p"]
matching_forms_count = 0
non_matching_forms_count = 0
no_predicted_forms_count = 0
non_matching_forms = {}
for i, row in tqdm.tqdm(enumerate(conflicts_data), total=len(conflicts_data)):
    sentence = row["sentence"]
    word = row["word_form"]
    word_span = row["span"]
    bert_Morph_v2_form = row["bert_morph_v2_form"]
    bmt_form, parsed_form = parse_bmt_form(bert_Morph_v2_form)
    vabamorf_form = row["current_case"]
    # print(f"Sentence: {sentence}")
    # print(f"Word: {word}, Span: {word_span}")
    # print(f"Bert Morph v2 form: {bert_Morph_v2_form}")
    # print(f"Vabamorf form: {vabamorf_form}")
    if parsed_form is None:
        no_predicted_forms_count += 1
    elif parsed_form == vabamorf_form:
        matching_forms_count += 1
    else:
        non_matching_forms_count += 1
        non_matching_forms[bmt_form] = non_matching_forms.get(bmt_form, 0) + 1

100%|██████████| 9634/9634 [00:00<00:00, 921356.33it/s]


In [106]:
non_matching_forms_with_np_count = non_matching_forms.get(
    "n", 0
) + non_matching_forms.get("p", 0)

print(
    f"Total forms count: {len(conflicts_data)} (100.00%) (Percentages are calculated based on the total number of forms in the dataset)"
)
print(
    f"Matching forms count: {matching_forms_count} ({matching_forms_count / len(conflicts_data) * 100:.2f}%)"
)
print(
    f"Non-matching forms count: {non_matching_forms_count} ({non_matching_forms_count / len(conflicts_data) * 100:.2f}%)"
)
print(
    f"No predicted forms count: {no_predicted_forms_count} ({no_predicted_forms_count / len(conflicts_data) * 100:.2f}%)"
)
print(
    f"Non-matching forms where Bert Morph v2 form is in ['n', 'p'] count: {non_matching_forms_with_np_count} ({non_matching_forms_with_np_count / len(conflicts_data) * 100:.2f}%)"
)
print(
    f"Non-matching forms where Bert Morph v2 form is 'g' count: {non_matching_forms['g']} ({non_matching_forms['g'] / len(conflicts_data) * 100:.2f}%)"
)
print("All non-matching forms counts:")
for form, count in non_matching_forms.items():
    print(f"  {form}: {count} ({count / len(conflicts_data) * 100:.2f}%)")

Total forms count: 9634 (100.00%) (Percentages are calculated based on the total number of forms in the dataset)
Matching forms count: 1551 (16.10%)
Non-matching forms count: 7712 (80.05%)
No predicted forms count: 371 (3.85%)
Non-matching forms where Bert Morph v2 form is in ['n', 'p'] count: 6931 (71.94%)
Non-matching forms where Bert Morph v2 form is 'g' count: 63 (0.65%)
All non-matching forms counts:
  n: 5202 (54.00%)
  p: 1729 (17.95%)
  all: 487 (5.06%)
  tr: 158 (1.64%)
  el: 25 (0.26%)
  es: 4 (0.04%)
  g: 63 (0.65%)
  ad: 13 (0.13%)
  ks: 1 (0.01%)
  ill: 6 (0.06%)
  in: 4 (0.04%)
  o: 7 (0.07%)
  kom: 7 (0.07%)
  ter: 4 (0.04%)
  ge: 1 (0.01%)
  abl: 1 (0.01%)


In [ ]:
for i, row in tqdm.tqdm(enumerate(conflicts_data), total=len(conflicts_data)):
    sentence = row["sentence"]
    word = row["word_form"]
    word_span = row["span"]
    bert_Morph_v2_form = row["bert_morph_v2_form"]
    bmt_form, parsed_form = parse_bmt_form(bert_Morph_v2_form)
    vabamorf_form = row["current_case"]
    # print(f"Sentence: {sentence}")
    # print(f"Word: {word}, Span: {word_span}")
    # print(f"Bert Morph v2 form: {bert_Morph_v2_form}")
    # print(f"Vabamorf form: {vabamorf_form}")
    if bmt_form in ["n", "p"]:
        print(f"Sentence: {sentence}")
        print(f"Word: {word}, Span: {word_span}")
        print(f"Bert Morph v2 form: {bert_Morph_v2_form}")
        print(f"Vabamorf form: {vabamorf_form}")

 19%|█▉        | 1844/9634 [00:00<00:00, 18251.08it/s]

Sentence: Kristina Shmiguni ahistas Holmenkollenis 30 km klassikatehnikasõidu lõpukilomeetritel kurnatus , ent viiendast kohast maailma karikasarja liidrirüü hoidmiseks piisas .
Word: Kristina, Span: [0, 8]
Bert Morph v2 form: ['sg n']
Vabamorf form: gen
Sentence: Tarkvara aitab jagu saada mitmes ametkonnas võidutsevast liigsest bürokraatiast .
Word: Tarkvara, Span: [0, 8]
Bert Morph v2 form: ['sg n']
Vabamorf form: gen
Sentence: Kui mu sõber aitab mul nii ja nii mitu head aktsiat omandada , mina aga aitan tal erastada teatud krundi , pole ju tegu sõprusega , vaid sõp-rust asendava tehinguga , seesuguse " sõprusega " , mille kohta Montesquieu ütles : " Tihtipeale on sõprus vaid leping , mis sunnib meid teisele väikest teenet osutama lootuses vastu saada midagi kopsakamat . "
Word: mina, Span: [63, 67]
Bert Morph v2 form: ['sg n']
Vabamorf form: gen
Sentence: Väheste kogemustega Buttonil aitab Monte Carlo raja saladusi tundma õppida endine vormelisõitja Gerhard Berger .
Word: Monte, Spa

 38%|███▊      | 3670/9634 [00:00<00:00, 10187.50it/s]

Word: Kristina, Span: [4, 12]
Bert Morph v2 form: ['sg n']
Vabamorf form: gen
Sentence: Kas Kristina Šmigun murrab lõpuks olümpianeeduse ja saab ometi kord medali ?
Word: Kristina, Span: [4, 12]
Bert Morph v2 form: ['sg n']
Vabamorf form: gen
Sentence: Doktor Rein Jalak helistas ja teatas , et Kristina on mootorsaanilt kukkudes rangluu murdnud .
Word: Kristina, Span: [42, 50]
Bert Morph v2 form: ['sg n']
Vabamorf form: gen
Sentence: « Isa hakkas tütardega tõsisemalt tegelema pärast seda , kui Kristina oli autoavariis reieluu murdnud ja üks jalg kippus teisest nõrgemaks jääma . »
Word: Kristina, Span: [61, 69]
Bert Morph v2 form: ['sg n']
Vabamorf form: gen
Sentence: Salt Lake City eel murdis Veerpalu jala ja Maed sakutas seljavalu .
Word: Veerpalu, Span: [26, 34]
Bert Morph v2 form: ['sg n']
Vabamorf form: gen
Sentence: Kui 70 km klassikamaratonist pool läbi sai , murdis Veerpalu võistlejaterivi tippu , et proovida koos mõne tugevamaga eest pageda .
Word: Veerpalu, Span: [52, 60]
Bert 

 65%|██████▌   | 6281/9634 [00:00<00:00, 9814.44it/s] 


Vabamorf form: gen
Sentence: Selle tähtsus ähvardab tulevikus veelgi kasvada , kui Saksa ja Vene selle põhja ühise gaasijuhtme panevad .
Word: Saksa, Span: [54, 59]
Bert Morph v2 form: ['sg n']
Vabamorf form: gen
Sentence: Praegu paneb raha asjad paika ja selle eest saab kõike , " ütles Sergei .
Word: raha, Span: [13, 17]
Bert Morph v2 form: ['sg n']
Vabamorf form: gen
Sentence: Teised , ennustades katastroofilist suurvett ja jääummistusi jõgedel , on pannud kolmandad lõhkama jääd .
Word: jääd, Span: [99, 103]
Bert Morph v2 form: ['sg p']
Vabamorf form: 
Sentence: Minu palvel pani pereema kirja oma perekonna kulumajanduse : Jaagupi koolis säästame toiduraha , sest lastel on koolitoit tasuta .
Word: pereema, Span: [17, 24]
Bert Morph v2 form: ['sg n']
Vabamorf form: gen
Sentence: Kord aastas paneb Liivi Soova keskaegsed rõivad selga ning suundub oma mõtetes ja tegudes sellesse aega , mis näib paljudele neile , kes keskaega vaid kooliõpikute vahendusel teavad , pime ja barbaarne .
Word:

100%|██████████| 9634/9634 [00:00<00:00, 11693.66it/s]

Word: Puhhi, Span: [62, 67]
Bert Morph v2 form: ['sg p']
Vabamorf form: gen
Sentence: Üleskutse põhiseadusliku võimu vägivaldsele kukutamisele teises riigis on piisav alus vajalike õiguslike meetmete rakendamiseks , " tsiteeris Lavrovi Interfax/BNS .
Word: Lavrovi, Span: [142, 149]
Bert Morph v2 form: ['sg p']
Vabamorf form: gen
Sentence: Mis täpselt , seda on keeruline öelda , » tsiteerib UPI Washingtoni Cato instituudi asepresidenti Ted Galen Carpenteri .
Word: asepresidenti, Span: [84, 97]
Bert Morph v2 form: ['sg p']
Vabamorf form: adit
Sentence: « Ma võlgnen oma naha kvaliteedi eest tänu oma ilukirurgile , » tsiteerib Gala kaunitari .
Word: Gala, Span: [74, 78]
Bert Morph v2 form: ['sg n']
Vabamorf form: gen
Sentence: Praegu tsiteerivad aga riigiraadio ja riigi-TV südamerahuga SLÕL-i ega teadvusta endale , mida see kõik ajakirjanduse mainele tähendab .
Word: SLÕL-i, Span: [60, 66]
Bert Morph v2 form: ['sg p']
Vabamorf form: gen
Sentence: Ja muist tuli välja ja muist ei tulnud mitt